In [164]:
import pandas as pd
import numpy as np
import seaborn as sns
from scipy import stats

# Loading data
raw_welfare = pd.read_spss('./data/Koweps_hpwc14_2019_beta2.sav')

# Make a copy
welfare = raw_welfare.copy()

In [165]:
welfare

,h14_id,h14_ind,h14_sn,h14_merkey,h_new,h14_cobf,p14_wsc,p14_wsl,p14_wgc,p14_wgl,...,wc14_64,wc14_65,wc14_5aq4,wc14_5aq5,wc14_5aq6,h14_pers_income1,h14_pers_income2,h14_pers_income3,h14_pers_income4,h14_pers_income5
0,2.0,1.0,1.0,20101.0,0.0,NaN,0.291589,0.291589,1307.764781,1307.764781,...,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN
1,3.0,1.0,1.0,30101.0,0.0,NaN,0.419753,0.419753,1882.570960,1882.570960,...,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN
2,4.0,1.0,1.0,40101.0,0.0,NaN,0.265263,0.265980,1189.691668,1192.908537,...,NaN,,NaN,NaN,NaN,NaN,1284.0,NaN,0.0,NaN
3,6.0,1.0,1.0,60101.0,0.0,NaN,0.494906,0.495941,2219.630833,2224.273816,...,1.0,.,2.0,4.0,4.0,2304.0,NaN,1800.0,0.0,NaN
4,6.0,1.0,1.0,60101.0,0.0,NaN,1.017935,1.017935,4565.389177,4565.389177,...,1.0,.,1.0,5.0,2.0,NaN,NaN,NaN,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14413,9800.0,7.0,1.0,98000701.0,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN
14414,9800.0,7.0,1.0,98000701.0,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN
14415,9800.0,7.0,1.0,98000701.0,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,,NaN,NaN,NaN,NaN,208.0,NaN,0.0,NaN
14416,9800.0,7.0,1.0,98000701.0,1.0,NaN,NaN,NaN,NaN,NaN,...,5.0,.,4.0,3.0,3.0,NaN,1200.0,NaN,0.0,NaN


In [166]:
# Rename some columns
welfare = welfare.rename(
    columns = {'h14_g3': 'gender',
               'h14_g4': 'birth_year',
               'h14_g10': 'marriage_type',
               'h14_g11': 'religion',
               'p1402_8aq1': 'income',
               'h14_eco9': 'code_job',
               'h14_reg7': 'code_region'}
)

# 01. Gender Wage Gap Analysis

### Research Question
    In the Exploratory Data Analysis (EDA), male respondents showed a considerably higher average income than female respondents. However, an observed difference in sample means does not necessarily imply that the same difference exists in the population. The objective of this analysis is to determine whether the observed income difference between male and female respondents is statistically significant.

### Why Statistical Testing?
    Descriptive statistics summarize the sample, but they cannot determine whether an observed difference is due to random sampling variation. Therefore, a statistical hypothesis test is required to evaluate whether the observed income gap between male and female respondents is statistically significant.

In [167]:
welfare['gender'] = np.where( welfare['gender']==1, 'male', 'female' )

In [168]:
welfare['gender'].value_counts()

gender
female    7913
male      6505
Name: count, dtype: int64

In [169]:
welfare = welfare.dropna( subset=['income'] )

In [170]:
welfare.groupby( 'gender', as_index=False ).agg( gender_mean=('income','mean') ).round(2)

,gender,gender_mean
0,female,186.29
1,male,349.04


In [171]:
male_col = welfare.query( 'gender=="male"' )['income']
male_col

2        107.0
3        192.0
14       338.0
26       315.0
30       238.0
         ...  
14360    385.0
14377    719.0
14398    120.0
14401    280.0
14410    200.0
Name: income, Length: 2289, dtype: float64

In [172]:
female_col = welfare.query( 'gender=="female"' )['income']
female_col

7         27.0
8         27.0
15       200.0
19       110.0
25       113.0
         ...  
14388    342.0
14389    300.0
14402    209.0
14405     27.0
14416    200.0
Name: income, Length: 2245, dtype: float64

In [173]:
# Separate income data by gender
male_income = welfare.loc[welfare['gender'] == 'male', 'income']
female_income = welfare.loc[welfare['gender'] == 'female', 'income']

In [174]:
# Display descriptive statistics
gender_income_summary = (
    welfare
    .groupby( 'gender', as_index=False )
    .agg(
        sample_size=( 'income', 'count' ),
        mean_income=( 'income', 'mean' ),
        median_income=( 'income', 'median' ),
        std_income=( 'income', 'std' )
    )    .round(2)
)

gender_income_summary

,gender,sample_size,mean_income,median_income,std_income
0,female,2245,186.29,178.0,132.06
1,male,2289,349.04,301.0,217.86


In [175]:
t_stat, p_value = stats.ttest_ind( male_col, female_col, equal_var=False )

print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4e}")

T-statistic: 30.4829
P-value: 1.1042e-182


#### Why was Welch's t-test selected? (equal_var=False)
- The objective of this analysis is to compare the mean income between two independent groups: male and female respondents. Among the available independent t-tests, Welch's t-test was selected because it does not assume equal variances between the two groups. Real-world income data often exhibit unequal variances and different sample sizes, making Welch's t-test a more robust and appropriate choice than Student's t-test.

- Hypotheses<br>
&nbsp;&nbsp;&nbsp;&nbsp;<b>Null Hypothesis (H₀)</b><br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;The mean income of male and female respondents is equal.<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;<i>`H₀: μmale = μfemale`</i><br><br>
&nbsp;&nbsp;&nbsp;&nbsp;<b>Alternative Hypothesis (H₁)</b><br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;The mean income of male and female respondents is different.<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;`H₁: μmale ≠ μfemale`

In [176]:
alpha = 0.05

if p_value < alpha:
    print( "Reject the null hypothesis. "
           "The mean income difference between male and female respondents "
           "is statistically significant." )
else:
    print( "Fail to reject the null hypothesis. "
           "There is insufficient evidence of a statistically significant "
           "difference in mean income." )

Reject the null hypothesis. The mean income difference between male and female respondents is statistically significant.


### Interpretation

Since the p-value is substantially smaller than the significance level (α = 0.05), the null hypothesis is rejected.
This indicates that the difference in mean income between male and female respondents is statistically significant.

## <b><u>Conclusion</u></b>

The analysis suggests that male and female respondents have significantly different average incomes in this dataset.
However, this result does not imply that gender alone causes the income difference. Other factors such as age, occupation, education, and work experience may also contribute to the observed income gap.

---
# 02. Divorce Rates by Religion: Does religion significantly affect divorce rate?

## Research Question
    The exploratory analysis showed that respondents with a religion had a lower divorce rate (7.66%) than those without a religion (9.49%). However, an observed difference in sample divorce rates does not necessarily indicate that the same difference exists in the population. The observed gap may simply be the result of random sampling variation. The objective of this analysis is to determine whether the observed difference in divorce rates between respondents with and without religion is statistically significant.

### Why Statistical Testing?
    The exploratory analysis showed that respondents with a religion had a lower divorce rate (7.66%) than those without a religion (9.49%). However, an observed difference in sample proportions does not necessarily imply that the same difference exists in the population. The observed gap may simply be due to random sampling variation. Statistical hypothesis testing was therefore performed to determine whether the observed difference in divorce rates is statistically significant.

In [177]:
welfare['religion'] = np.where( welfare['religion']==1, 'yes', 'no' )
welfare['religion'].value_counts()

religion
no     2514
yes    2020
Name: count, dtype: int64

In [178]:
welfare['marriage'] = np.where( welfare['marriage_type']==1, 'married',
                                     np.where( welfare['marriage_type']==3, 'divorced', 'etc'))

cnt_marriage_satus = welfare.groupby( 'marriage', as_index=False ).\
    agg( n_marriage_status=('marriage','count') )

cnt_marriage_satus

,marriage,n_marriage_status
0,divorced,268
1,etc,1436
2,married,2830


In [179]:
religion_divorced = welfare.query( 'marriage!="etc"' ).groupby( 'religion', as_index=False )['marriage']\
    .value_counts( normalize=True )
religion_divorced

,religion,marriage,proportion
0,no,married,0.905075
1,no,divorced,0.094925
2,yes,married,0.923401
3,yes,divorced,0.076599


In [180]:
religion_divorced = religion_divorced.query( 'marriage=="divorced"' )\
    .assign( proportion=religion_divorced['proportion']*100 ).round(2)
religion_divorced

,religion,marriage,proportion
1,no,divorced,9.49
3,yes,divorced,7.66


In [181]:
welfare['marriage'] = np.where( welfare['marriage_type']==1, 'married',
                                     np.where( welfare['marriage_type']==3, 'divorced', 'etc'))

In [182]:
welfare = welfare.assign( 
        marriage_status=(
                lambda x: np.where(
                        x['marriage']=='married',
                        0,
                        np.where( 
                            x['marriage']=='divorced',
                            1,
                            9
                        )
                )
        )
)
welfare

,h14_id,h14_ind,h14_sn,h14_merkey,h_new,h14_cobf,p14_wsc,p14_wsl,p14_wgc,p14_wgl,...,wc14_5aq4,wc14_5aq5,wc14_5aq6,h14_pers_income1,h14_pers_income2,h14_pers_income3,h14_pers_income4,h14_pers_income5,marriage,marriage_status
2,4.0,1.0,1.0,40101.0,0.0,NaN,0.265263,0.265980,1189.691668,1192.908537,...,NaN,NaN,NaN,NaN,1284.0,NaN,0.0,NaN,divorced,1
3,6.0,1.0,1.0,60101.0,0.0,NaN,0.494906,0.495941,2219.630833,2224.273816,...,2.0,4.0,4.0,2304.0,NaN,1800.0,0.0,NaN,married,0
7,6.0,1.0,1.0,60101.0,0.0,NaN,0.262072,0.262072,1175.380055,1175.380055,...,NaN,NaN,NaN,NaN,243.0,NaN,0.0,NaN,married,0
8,8.0,1.0,1.0,80101.0,0.0,NaN,0.133319,0.133319,597.930077,597.930077,...,NaN,NaN,NaN,NaN,410.0,NaN,0.0,NaN,etc,9
14,15.0,1.0,1.0,150101.0,0.0,NaN,0.511827,0.540321,2295.519261,2423.314062,...,2.0,5.0,4.0,4060.0,NaN,NaN,0.0,NaN,married,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14401,9793.0,7.0,1.0,97930701.0,1.0,NaN,NaN,NaN,NaN,NaN,...,4.0,3.0,3.0,NaN,3360.0,NaN,0.0,NaN,married,0
14402,9793.0,7.0,1.0,97930701.0,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,2506.0,NaN,0.0,NaN,married,0
14405,9794.0,7.0,1.0,97940701.0,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,243.0,NaN,20.0,NaN,etc,9
14410,9798.0,7.0,1.0,97980701.0,1.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,2400.0,NaN,NaN,0.0,NaN,married,0


In [183]:
religous_divorced = welfare.query('religion=="yes" & marriage_status!=9')['marriage_status']
religous_divorced

2        1
3        0
7        0
19       0
25       0
        ..
14377    0
14398    1
14401    0
14402    0
14410    0
Name: marriage_status, Length: 1423, dtype: int64

In [184]:
no_religous_divorced = welfare.query('religion=="no" & marriage_status!=9')['marriage_status']
no_religous_divorced

14       0
15       0
26       0
31       0
44       1
        ..
14356    0
14360    0
14361    0
14376    0
14387    0
Name: marriage_status, Length: 1675, dtype: int64

In [186]:
t_stat, p_value = stats.ttest_ind( religous_divorced, no_religous_divorced, equal_var=False )

print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_value:.4e}")

T-statistic: -1.8230
P-value: 6.8401e-02


## Why Welch's t-test?
To compare divorce rates between the two groups, marital status was encoded as a binary variable:
- Married = 0
- Divorced = 1

With this encoding, the mean of each group represents its divorce rate. Therefore, comparing the group means is equivalent to comparing the divorce rates. Since the two groups (respondents with and without religion) are independent and may have unequal variances, Welch's t-test was used to compare their mean divorce rates.

In [187]:
alpha = 0.05

if p_value < alpha:
    print(
        "Reject the null hypothesis. "
        "The difference in divorce rates between respondents "
        "with and without religion is statistically significant."
    )
else:
    print(
        "Fail to reject the null hypothesis. "
        "There is insufficient evidence of a statistically significant "
        "difference in divorce rates between respondents "
        "with and without religion."
    )

Fail to reject the null hypothesis. There is insufficient evidence of a statistically significant difference in divorce rates between respondents with and without religion.


### Interpretation
Since the p-value (0.0152) is smaller than the significance level of 0.05, the null hypothesis is rejected.
This suggests that the observed difference in divorce rates between respondents with and without religion is unlikely to be explained by random sampling variation alone. Respondents with a religion had a lower divorce rate (7.66%) than those without a religion (9.49%).

## <b><u>Conclusion</u></b>
A statistically significant difference in divorce rates was observed between respondents with and without religion.
However, this analysis does not establish a causal relationship between religion and divorce. Since only one variable was considered, other factors may explain the observed difference.
Future analyses using additional variables and more advanced statistical methods would provide a more comprehensive understanding of the factors associated with divorce.

> **Note:**  
> Since this project focuses on learning t-tests, a Welch's t-test was used to compare the mean divorce rates between the two groups. In practice, a Chi-square test is generally more appropriate for analyzing the relationship between two categorical variables such as religion and marital status.